# Agentowy asystent RAG

Przeżyłeś katastrofę Titanica i lądujesz na bezludnej wyspie. Masz przy sobie tylko teczkę z **plikami PDF** i terminal. Agent z zewnątrz zrzuca na ciebie odpowiedzialność za operację – musisz to ogarnąć. Zbuduj **asystenta**, który przejmie zadania techniczne i zagwarantuje ci wyłącznie sprawdzone odpowiedzi. Mechanizm jest prosty: system ma korzystać stricte z ocalałych dokumentów, całkowicie odciąć **halucynacje** i od razu zgłaszać brak wiedzy, jeśli danej informacji nie ma w plikach.

### A. Co budujesz

Asystenta **agentowego**, nie klasycznego RAG-a. Klasyczny RAG to sztywno ustalony przebieg postępowania:

(`embed → pobierz → wklej kontekst → generuj`)

 — zawsze te same kroki, żadnej decyzji.

**Agent** sam decyduje,
czy, kiedy i którego narzędzia użyć, ogląda wynik i może wykonać kolejny krok.

### B. Proponowana zawężona tematyka PDF

Zbierz **3–5 PDF-ów na jeden, spójny temat** i wrzuć do folderu.

**Przepisy kulinarne** — trzeba nakarmić innych rozbitków. Przykładowe drugie narzędzie:
przeliczanie porcji, lista zakupów z kilku przepisów, porównanie z zawartością spiżarni, sumowanie czasu przygotowania.

**Instrukcje obsługi** (jakieś przedmioty przydatne do przetrwania). Przykładowe drugie narzędzie:
zużycie paliwa, przelicznik jednostek (PSI ↔ bar, °F ↔ °C), harmonogram serwisu, wyszukiwarka kodów błędów.

Drugie narzędzie ma mieć **weryfikowalne wyjście** — takie, po którym widać, że policzyło, a nie zgadło.
Narzędzie, które tylko przepisuje tekst albo którego agent nigdy nie wywoła, się nie liczy.

### C. Wybór narzędzi

| Warstwa | Czego najlepiej użyć |
|---|---
| Parsowanie PDF | PyMuPDF
| Podział na fragmenty | `RecursiveCharacterTextSplitter` |
| Embeddingi | model z HuggingFace |
| Baza wektorowa | Chroma lub Qdrant |
| Model generatywny | darmowe API z **natywnym tool callingiem** (Groq, Gemini) |
| Agent | `create_agent` z LangChain |
| Pamięć | `InMemorySaver` z LangGraph |


### D. Polecenie:

- Załaduj 5 dokumentów PDF dotyczących wybranej tematyki, podział na strony, umieść je w folderze na dysku
- Pamiętaj o kluczu do LLM, trzymaj go w folderze `../.env`
- Podziel strony na chunki
- Wybierz model do embeddingów z HuggingFace
- Utwórz bazę wektorową i wypełnij ją danymi
- Zdefiniuj **co najmniej** 2 narzędzia
- Zbuduj agenta, który opiera się na ustalonym prompcie systemowym
- Sprawdź jak działa, zadając pytania (nie)dotyczące dokumentów z bazy

**Dodatkowe wymagania:**
- Mechanizm pamięci konwersacji - tak, żeby model pamiętał poprzednią część konwersacji
- Mechanizm braku halucynacji - jeśli model nie znajdzie odpowiedzi w bazie, informuje o tym
- Cytowanie dokumentu, z którego model pobrał informacje

**Dla chętnych:**
- Model-sędzia (LLM-as-judge) - ocena generowanych odpowiedzi (faithfulness, answer relevancy)
- Retrieval metrics: hit rate@k, MRR
- Rewriting pytań zależnych od kontekstu (zaimki) przed retrievalem
- Lepsze czyszczenie PDF (OCR, czyszczenie), tuning chunk_size/chunk_overlap

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza Markdown.

Jeśli chodzi o wybór pdfów, to nie udało mi się znaleźć w Internecie kilku spójnych PDF-ów z przepisami w odpowiednim formacie, dlatego wygenerowałam zestaw dokumentów przy pomocy AI.

### 1. Środowisko i dane

In [49]:
from dotenv import load_dotenv
load_dotenv()

True

## Parsowanie PDF
Parsuję wszystkie PDF-y i łączę ich zawartość w jeden tekst.

In [50]:
import fitz

def load_pdf(pdf_path):

    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document):

        pages.append({
            "text": page.get_text(),
            "source": pdf_path,
            "page": page_number + 1
        })

    document.close()

    return pages

In [51]:
zupy = load_pdf("files/zupy_i_gulasze.pdf")
ryby = load_pdf("files/ryby_i_dziczyzna.pdf")
rosliny = load_pdf("files/rosliny_i_grzyby.pdf")
pieczenie = load_pdf("files/pieczenie_na_ognisku.pdf")
desery = load_pdf("files/desery_i_napoje.pdf")

In [52]:
documents = zupy + ryby + rosliny + pieczenie + desery

In [53]:
print(documents[0]["text"][:500])

Kuchnia Ogniskowa
Zupy i gulasze
Wstęp
Przepisy z tego zbioru zostały przygotowane z myślą o gotowaniu na ognisku lub kuchence 
turystycznej. Wykorzystują produkty, które można zabrać do obozu lub znaleźć w lesie. Wszystkie 
receptury można przygotować przy użyciu jednego garnka lub żeliwnego kociołka.
1. Zupa z pokrzywy i ziemniaków
Opis
Pokrzywa jest bogata w witaminy i minerały. Najlepiej zbierać młode liście, zanim roślina 
zakwitnie.
Składniki
•
2 garście młodej pokrzywy
•
3 średnie ziemnia


In [54]:
## Podział na chunki

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [56]:
chunks = []

for document in documents:

    text = document["text"]

    if "1." in text:
        text = text[text.find("1."):]

    split_texts = text_splitter.split_text(text)

    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"]
        })

In [57]:
print(len(chunks))

27






## Wybór modelu embeddingów
Do projektu wybrałam model `sentence-transformers/all-MiniLM-L6-v2`, ponieważ jest darmowy, łatwy w użyciu i często wykorzystywany w prostych systemach RAG. Model zamienia tekst na wektory, dzięki czemu możliwe jest wyszukiwanie podobnych fragmentów dokumentów.


In [58]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8693.47it/s]


## Zamiana chunków na obiekty Document

In [59]:
from langchain_core.documents import Document

langchain_documents = []

for chunk in chunks:
    langchain_documents.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk["source"],
                "page": chunk["page"]
            }
        )
    )

## Utworzenie bazy wektorowej

In [60]:
from langchain_chroma import Chroma

In [61]:
vector_db = Chroma.from_documents(
    documents=langchain_documents,
    embedding=embedding_model,
    persist_directory="./baza"
)

In [62]:
print(vector_db._collection.count())

27


## Test wyszukiwania

Sprawdzam, czy baza wektorowa zwraca odpowiednie fragmenty dokumentów dla przykładowego zapytania.

In [63]:
results = vector_db.similarity_search(
    "placek z ziołami",
    k=5
)

for result in results:
    print(result.metadata)
    print(result.page_content[:300])
    print("-" * 50)

{'page': 2, 'source': 'files/pieczenie_na_ognisku.pdf'}
Opis
Szybki placek z dodatkiem świeżych ziół.
Składniki
•
300 g mąki pszennej
•
250 ml wody
•
2 łyżki oliwy
•
szczypiorek
•
koperek
•
sól
•
pieprz
Sposób przygotowania
--------------------------------------------------
{'source': 'files/pieczenie_na_ognisku.pdf', 'page': 3}
Miękkie bułeczki pieczone w garnku ustawionym nad żarem.
Składniki
•
500 g mąki pszennej
•
7 g suchych drożdży
•
300 ml mleka
•
50 g masła
•
1 jajko
--------------------------------------------------
{'source': 'files/pieczenie_na_ognisku.pdf', 'page': 2}
1.
Połącz suche składniki.
2.
Dodaj wodę i olej.
3.
Wyrób ciasto.
4.
Uformuj płaski placek.
5.
Piecz na patelni lub kamieniu około 10 minut z każdej strony.
Czas przygotowania: 30 minut
Liczba porcji: 4
3. Pieczone ziemniaki z masłem ziołowym
Opis
Klasyczne danie przygotowywane bezpośrednio w żarze.
--------------------------------------------------
{'source': 'files/zupy_i_gulasze.pdf', 'page': 1}
1. Zupa z p

Baza wektorowa poprawnie wyszukuje fragmenty dokumentów najbardziej zbliżone do zapytania użytkownika.

In [64]:
print(len(chunks))
print(vector_db._collection.count())

27
27


## Wczytanie klucza z .env

## Model językowy
Do generowania odpowiedzi wykorzystuję model Gemini. Będzie on korzystał z przygotowanych narzędzi zamiast odpowiadać wyłącznie na podstawie własnej wiedzy.

In [65]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
)

In [66]:
response = llm.invoke("Odpowiedz jednym słowem: test")

print(response.content)

[{'type': 'text', 'text': 'Test', 'extras': {'signature': 'EnEKbwERTTIPiaND+DGuxTEzO/eTkN9mpUhDkr52fP95fFNptXRvxpyXZw8KTlSy0I/39EqW7InmZrqL01rcVKH+YgHhHEq/1LLLdFqYUcavFOv+b8+AWphHQg+SMoobkoC93j2vUWX7RqkU3OBC0tsi5g=='}}]


In [67]:
from langchain.tools import tool

@tool
def search_recipes(query: str) -> str:
    """Wyszukuje informacje o przepisach w bazie dokumentów PDF."""

    results = vector_db.similarity_search(query, k=3)

    if not results:
        return "Nie znaleziono informacji."

    answer = ""

    for doc in results:
        answer += (
            f"Źródło: {doc.metadata['source']}, strona {doc.metadata['page']}\n"
            f"{doc.page_content}\n\n"
        )

    return answer

In [68]:
from langchain.tools import tool
import re


@tool
def scale_recipe(recipe_text: str, factor: float) -> str:
    """
    Przelicza ilości składników w przepisie dla podanego współczynnika.
    """
    lines = recipe_text.split("\n")
    result = []

    in_ingredients = False

    pattern = r"(\d+(?:[.,]\d+)?)"

    for line in lines:
        if "Składniki" in line:
            in_ingredients = True
            result.append(line)
            continue

        if "Sposób przygotowania" in line:
            in_ingredients = False
            result.append(line)
            continue

        if in_ingredients:
            match = re.search(pattern, line)

            if match:
                value = float(match.group(1).replace(",", "."))
                new_value = round(value * factor, 2)

                line = line.replace(match.group(1), str(new_value), 1)

        result.append(line)

    return "\n".join(result)

In [69]:
system_prompt = """
Jesteś agentowym asystentem kulinarnym.

Masz dostęp wyłącznie do narzędzi.

Do wyszukiwania informacji używaj narzędzia search_recipes.

Do przeliczania składników używaj narzędzia scale_recipe.

Nigdy nie odpowiadaj z własnej wiedzy.

Jeżeli informacji nie ma w dokumentach, odpowiedz:
'Nie znalazłem informacji w dostępnych dokumentach.'

Każdą odpowiedź zakończ podaniem źródła oraz numeru strony.
"""

In [70]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[search_recipes, scale_recipe],
    system_prompt=system_prompt,
    checkpointer=memory
)

## Test halucynacji

In [71]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Jak zrobić pizzę hawajską?"}]},
    config=config
)

print(response["messages"][-1].content[0]["text"])

Nie znalazłem informacji w dostępnych dokumentach. 

Źródło: brak informacji w dokumentach.


Agent nie odpowiada z własnej wiedzy. Jeśli informacja nie występuje w dokumentach, informuje o jej braku.





## Test pamięci

In [77]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Pokaż przepis na pieczone ziemniaki z masłem ziołowym."}]},
    config=config
)

print(response["messages"][-1].content[0]["text"])

Nie znalazłem informacji o przepisie na "podpiekane ziemniaczki z masełkiem". Dostępny jest natomiast przepis na pieczone ziemniaki z masłem ziołowym (strona 2, plik `pieczenie_na_ognisku.pdf`) oraz przepis na ziemniaki podawane z masłem i szczypiorkiem (strona 5, plik `ryby_i_dziczyzna.pdf`).

Źródło: files/pieczenie_na_ognisku.pdf, strona 2 oraz files/ryby_i_dziczyzna.pdf, strona 5


In [77]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Pokaż przepis na podpiekane ziemniaczki z masełkiem."}]},
    config=config
)

print(response["messages"][-1].content[0]["text"])

Nie znalazłem informacji o przepisie na "podpiekane ziemniaczki z masełkiem". Dostępny jest natomiast przepis na pieczone ziemniaki z masłem ziołowym (strona 2, plik `pieczenie_na_ognisku.pdf`) oraz przepis na ziemniaki podawane z masłem i szczypiorkiem (strona 5, plik `ryby_i_dziczyzna.pdf`).

Źródło: files/pieczenie_na_ognisku.pdf, strona 2 oraz files/ryby_i_dziczyzna.pdf, strona 5


Zmodyfikowałam trochę poprzednie polecenie, aby sprawdzić co w sytuacji, gdy w poleceniu nie użyję dokładnej nazwy przepisu z pliku.

In [73]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Na ile porcji jest ten przepis?"}]},
    config=config
)

print(response["messages"][-1].content[0]["text"])

Ten przepis jest przewidziany na 4 porcje.

Źródło: files/pieczenie_na_ognisku.pdf, strona 2


Mechanizm pamięci działa poprawnie. Agent pamięta wcześniejsze wiadomości w ramach tej samej konwersacji.


## Test narzędzia do przeliczania liczby porcji

In [74]:
recipe = search_recipes.invoke("placek ziołowy")

print(recipe)

Źródło: files/pieczenie_na_ognisku.pdf, strona 2
Opis
Szybki placek z dodatkiem świeżych ziół.
Składniki
•
300 g mąki pszennej
•
250 ml wody
•
2 łyżki oliwy
•
szczypiorek
•
koperek
•
sól
•
pieprz
Sposób przygotowania

Źródło: files/pieczenie_na_ognisku.pdf, strona 2
1.
Połącz suche składniki.
2.
Dodaj wodę i olej.
3.
Wyrób ciasto.
4.
Uformuj płaski placek.
5.
Piecz na patelni lub kamieniu około 10 minut z każdej strony.
Czas przygotowania: 30 minut
Liczba porcji: 4
3. Pieczone ziemniaki z masłem ziołowym
Opis
Klasyczne danie przygotowywane bezpośrednio w żarze.
Składniki
•
6 dużych ziemniaków
•
100 g masła
•
szczypiorek
•
natka pietruszki
•
sól
•
pieprz
Sposób przygotowania
1.
Dokładnie umyj ziemniaki.
2.
Umieść je w gorącym żarze.
3.
Piecz około 45 minut.
4.
Wymieszaj masło z posiekanymi ziołami.
5.
Podawaj gorące ziemniaki z masłem.
Czas przygotowania: 50 minut
Liczba porcji: 4
4. Placek ziołowy na patelni
Opis
Szybki placek z dodatkiem świeżych ziół.
Składniki
•
300 g mąki pszennej


In [75]:
print(scale_recipe.invoke({"recipe_text": recipe, "factor": 2}))

Źródło: files/pieczenie_na_ognisku.pdf, strona 2
Opis
Szybki placek z dodatkiem świeżych ziół.
Składniki
•
600.0 g mąki pszennej
•
500.0 ml wody
•
4.0 łyżki oliwy
•
szczypiorek
•
koperek
•
sól
•
pieprz
Sposób przygotowania

Źródło: files/pieczenie_na_ognisku.pdf, strona 2
1.
Połącz suche składniki.
2.
Dodaj wodę i olej.
3.
Wyrób ciasto.
4.
Uformuj płaski placek.
5.
Piecz na patelni lub kamieniu około 10 minut z każdej strony.
Czas przygotowania: 30 minut
Liczba porcji: 4
3. Pieczone ziemniaki z masłem ziołowym
Opis
Klasyczne danie przygotowywane bezpośrednio w żarze.
Składniki
•
12.0 dużych ziemniaków
•
200.0 g masła
•
szczypiorek
•
natka pietruszki
•
sól
•
pieprz
Sposób przygotowania
1.
Dokładnie umyj ziemniaki.
2.
Umieść je w gorącym żarze.
3.
Piecz około 45 minut.
4.
Wymieszaj masło z posiekanymi ziołami.
5.
Podawaj gorące ziemniaki z masłem.
Czas przygotowania: 50 minut
Liczba porcji: 4
4. Placek ziołowy na patelni
Opis
Szybki placek z dodatkiem świeżych ziół.
Składniki
•
600.0 g m

In [76]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Pokaż przepis na placek ziołowy i przelicz składniki na 8 porcji."}]},
    config=config)

print(response["messages"][-1].content[0]["text"])

Oto przepis na placek ziołowy (oryginalnie na 4 porcje), a poniżej przeliczenie składników na 8 porcji.

**Oryginalny przepis (na 4 porcje):**
* 300 g mąki pszennej
* 250 ml wody
* 2 łyżki oliwy
* szczypiorek
* koperek
* sól
* pieprz

**Sposób przygotowania:**
1. Połącz suche składniki.
2. Dodaj wodę i olej.
3. Wyrób ciasto.
4. Uformuj płaski placek.
5. Piecz na patelni lub kamieniu około 10 minut z każdej strony.

**Składniki przeliczone na 8 porcji:**
* 600.0 g mąki pszennej
* 500.0 ml wody
* 4.0 łyżki oliwy
* szczypiorek
* koperek
* sól
* pieprz

Źródło: files/pieczenie_na_ognisku.pdf, strona 2


Drugie narzędzie poprawnie przelicza ilości składników zgodnie z podanym współczynnikiem.
